# Retos del workshop RAG — mi copia

Resuelve **C1** (HNSW vs fuerza bruta con FAISS). C4 y C6 viven en el notebook 03 (necesitan el `retriever`/Qdrant de ese notebook); aquí dejo las piezas autónomas de C1.

Las conclusiones y tablas finales están en `RESPUESTAS.md`.

## C1 · HNSW contra fuerza bruta con FAISS

Línea base del starter + TODOs completados: gráficas y barrido de `n`.

In [ ]:
%pip install -q faiss-cpu matplotlib


In [ ]:
import time
import faiss
import numpy as np

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

def experimento(n, d=64, nq=200, k=10, Ms=(8, 16, 32),
                efs=(10, 16, 32, 64, 128, 256), seed=42):
    rng = np.random.default_rng(seed)
    xb = rng.random((n, d), dtype="float32")
    xq = rng.random((nq, d), dtype="float32")

    flat = faiss.IndexFlatL2(d); flat.add(xb)
    t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

    rows = []
    for M in Ms:
        index = faiss.IndexHNSWFlat(d, M)
        index.hnsw.efConstruction = 100
        t0 = time.time(); index.add(xb); build_s = time.time() - t0
        for ef in efs:
            index.hnsw.efSearch = ef
            t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
            rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))
    return t_flat, rows

t_flat, rows = experimento(20_000)
print(f"exacto: {t_flat:.4f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@10={rec:.3f}  {ms:.4f} ms/q  (build {b:.1f}s)")


In [ ]:
# TODO 1 resuelto: recall vs efSearch y recall vs latencia
import matplotlib.pyplot as plt

efs = (10, 16, 32, 64, 128, 256)
por_M = {M: ([], []) for M in (8, 16, 32)}
for M, ef, rec, ms, b in rows:
    por_M[M][0].append(rec); por_M[M][1].append(ms)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for M, (recs, lats) in por_M.items():
    ax[0].plot(efs, recs, marker="o", label=f"M={M}")
    ax[1].plot(lats, recs, marker="s", label=f"M={M}")
ax[0].axhline(0.95, ls="--", color="gray"); ax[0].set_xlabel("efSearch"); ax[0].set_ylabel("recall@10"); ax[0].set_title("recall vs efSearch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].axvline(t_flat, ls=":", color="red", label=f"exacto {t_flat:.3f} ms/q"); ax[1].set_xlabel("ms/consulta"); ax[1].set_ylabel("recall@10"); ax[1].set_title("recall vs latencia"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# TODO 2 resuelto: barrido de n. ¿Desde qué n HNSW supera al exacto?
for n in (20_000, 100_000, 500_000):
    t_flat_n, rows_n = experimento(n)
    hnsw_ms = [ms for *_, ms, _ in rows_n]
    print(f"n={n:>7,}: exacto {t_flat_n:.4f} ms/q | HNSW {min(hnsw_ms):.4f}–{max(hnsw_ms):.4f} ms/q")
    best16 = max((r for r in rows_n if r[0] == 16), key=lambda r: r[2])
    print(f"           mejor recall M=16: {best16[2]:.3f} (ef={best16[1]})")


## C1 · Observaciones (resumen; detalle en RESPUESTAS.md)

- Con **M=16** se alcanza **recall@10 ≥ 0.95** solo con **efSearch=256** en n=20 000; al crecer n el recall cae (datos aleatorios uniformes no tienen estructura de baja dimensión que el grafo pueda explotar).
- HNSW ya es **más rápido que el exacto desde n=20 000** y la ventaja se dispara en n=500 000 (exacto ≈ 1.9 ms/q vs HNSW ≤ 0.18 ms/q).
- Con **embeddings reales de texto** (intrínsecamente de baja dimensión efectiva) se espera mucho mejor recall al mismo ef, porque los vecinos reales están concentrados y el grafo de navegación acierta antes.